# RLHF Pipeline Walkthrough

This notebook walks through the full **Reinforcement Learning from Human Feedback (RLHF)** pipeline implemented in RewardLab.

## Pipeline Overview

```
1. Collect Preferences  →  2. Train Reward Model  →  3. PPO Fine-Tuning  →  4. Evaluate
```

RLHF aligns language models with human preferences by:
1. Collecting pairwise preference data (which response is better?)
2. Training a reward model to predict human preferences
3. Optimizing the policy using RL (PPO) to maximize reward
4. Keeping the policy from drifting too far with a KL penalty


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import yaml

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 1: Load Configuration

In [ ]:
with open('../configs/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

print('Configuration:')
for section, values in cfg.items():
    print(f'  [{section}]')
    if isinstance(values, dict):
        for k, v in values.items():
            print(f'    {k}: {v}')

## Step 2: Preference Data Collection

We generate preference pairs by:
1. Sampling prompts from templates
2. Generating two responses per prompt (different temperatures)
3. Scoring with a heuristic (length + diversity + relevance)
4. Labeling higher-scored as `chosen`, lower as `rejected`

In [ ]:
from data.preference_collector import PreferenceDataCollector, compute_heuristic_score

# Demonstrate the scoring function
prompt = 'Explain machine learning in simple terms.'
response_good = (
    'Machine learning is a type of artificial intelligence where computers learn patterns '
    'from data rather than being explicitly programmed. Instead of following rigid rules, '
    'the system improves its performance on tasks like image recognition, language translation, '
    'and recommendation systems by analyzing examples. The key idea is generalization: '
    'learning from a training set and applying knowledge to new, unseen data.'
)
response_poor = 'ML is when a computer does learning things with data.'

score_good = compute_heuristic_score(response_good, prompt)
score_poor = compute_heuristic_score(response_poor, prompt)

print(f'Good response score:  {score_good:.4f}')
print(f'Poor response score:  {score_poor:.4f}')
print(f'Correct labeling: {"chosen=good" if score_good > score_poor else "ERROR"}')

## Step 3: Dataset Loading

The `PreferenceDataset` loads JSONL preference data and tokenizes prompt+response pairs.

In [ ]:
import json
import os

# Check if data exists, otherwise create a tiny synthetic set for demo
data_path = '../data/preferences.jsonl'

if not os.path.exists(data_path):
    print('Creating synthetic demo data...')
    os.makedirs('../data', exist_ok=True)
    demo_pairs = [
        {
            'prompt': 'What is gradient descent?',
            'chosen': 'Gradient descent is an optimization algorithm that iteratively moves in the direction of steepest descent of a loss function to find its minimum.',
            'rejected': 'It is when the gradient goes down.'
        },
        {
            'prompt': 'Explain neural networks.',
            'chosen': 'Neural networks are computing systems inspired by biological brains, consisting of interconnected nodes that process information using connectionist approaches.',
            'rejected': 'Neural networks are like brains but in computers.'
        }
    ] * 25  # Repeat to have 50 pairs
    with open(data_path, 'w') as f:
        for pair in demo_pairs:
            f.write(json.dumps(pair) + '\n')
    print(f'Created {len(demo_pairs)} demo pairs at {data_path}')
else:
    with open(data_path) as f:
        n = sum(1 for _ in f)
    print(f'Found {n} preference pairs at {data_path}')

In [ ]:
from data.dataset import PreferenceDataset, create_dataloaders

train_loader, val_loader = create_dataloaders(
    data_path=data_path,
    tokenizer_name='gpt2',
    max_length=256,
    batch_size=4,
    val_split=0.2,
)

# Inspect a batch
batch = next(iter(train_loader))
print('Batch keys:', list(batch.keys()))
print('chosen_input_ids shape:', batch['chosen_input_ids'].shape)
print('rejected_input_ids shape:', batch['rejected_input_ids'].shape)

## Step 4: Reward Model

The `RewardModel` uses GPT-2 as a backbone with a linear head to output a scalar reward.

**Bradley-Terry loss:**
$$\mathcal{L} = -\log \sigma(r_{\text{chosen}} - r_{\text{rejected}})$$

This maximizes the probability that the chosen response has a higher reward than the rejected one.

In [ ]:
from models.reward_model import RewardModel

reward_model = RewardModel(model_name='gpt2', dropout=0.1)
reward_model.to(device)

n_params = sum(p.numel() for p in reward_model.parameters())
print(f'RewardModel parameters: {n_params:,}')
print(f'Backbone: {reward_model.config.model_type}')
print(f'Hidden size: {reward_model.config.hidden_size}')

In [ ]:
# Test forward pass
batch = next(iter(train_loader))
chosen_ids = batch['chosen_input_ids'].to(device)
chosen_mask = batch['chosen_attention_mask'].to(device)
rejected_ids = batch['rejected_input_ids'].to(device)
rejected_mask = batch['rejected_attention_mask'].to(device)

with torch.no_grad():
    r_chosen = reward_model(chosen_ids, chosen_mask)
    r_rejected = reward_model(rejected_ids, rejected_mask)

print(f'Chosen rewards:   {r_chosen.tolist()}')
print(f'Rejected rewards: {r_rejected.tolist()}')
print(f'Accuracy (before training): {(r_chosen > r_rejected).float().mean().item():.3f}')

In [ ]:
# Compute Bradley-Terry loss
loss, metrics = reward_model.compute_reward_loss(
    chosen_ids, chosen_mask, rejected_ids, rejected_mask
)
print('Loss metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## Step 5: Training the Reward Model (mini demo)

Here we run a few training steps to show the loss decreasing.

In [ ]:
import torch.nn as nn
from torch.optim import AdamW

optimizer = AdamW(reward_model.parameters(), lr=1e-5)

train_losses = []
train_accs = []

reward_model.train()
n_demo_steps = 20

for step, batch in enumerate(train_loader):
    if step >= n_demo_steps:
        break
    
    chosen_ids = batch['chosen_input_ids'].to(device)
    chosen_mask = batch['chosen_attention_mask'].to(device)
    rejected_ids = batch['rejected_input_ids'].to(device)
    rejected_mask = batch['rejected_attention_mask'].to(device)
    
    optimizer.zero_grad()
    loss, metrics = reward_model.compute_reward_loss(
        chosen_ids, chosen_mask, rejected_ids, rejected_mask
    )
    loss.backward()
    nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
    optimizer.step()
    
    train_losses.append(metrics['loss'])
    train_accs.append(metrics['accuracy'])

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='#2196F3', linewidth=2)
ax1.set_title('Training Loss (Bradley-Terry)', fontsize=12)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, color='#4CAF50', linewidth=2)
ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax2.set_title('Pairwise Accuracy', fontsize=12)
ax2.set_xlabel('Step')
ax2.set_ylabel('Accuracy')
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Reward Model Training (Demo)', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Final loss: {train_losses[-1]:.4f}')
print(f'Final accuracy: {train_accs[-1]:.4f}')

## Step 6: Policy Model and PPO

The `PolicyModel` wraps a causal LM and provides:
- `generate_with_logprobs`: generation + per-token log probabilities
- `compute_kl_divergence`: KL(policy || reference) for penalty computation

**PPO Objective:**
$$\mathcal{L}^{\text{PPO}} = -\min\left(\frac{\pi_\theta(a|s)}{\pi_{\theta_{\text{old}}}(a|s)} \hat{A}, \text{clip}\left(\frac{\pi_\theta}{\pi_{\text{old}}}, 1\pm\epsilon\right) \hat{A}\right) + \beta \cdot \text{KL}(\pi_\theta || \pi_{\text{ref}})$$

In [ ]:
from models.policy_model import PolicyModel

policy = PolicyModel(model_name='gpt2', device=device)

# Generate a response with log probabilities
prompt = 'Explain the concept of neural networks in simple terms.'
response, response_ids, log_probs = policy.generate_with_logprobs(
    prompt,
    max_new_tokens=50,
    temperature=0.8,
)

print('Prompt:', prompt)
print('Response:', response)
print(f'Response length: {len(response_ids)} tokens')
print(f'Log probs shape: {log_probs.shape}')
print(f'Avg log prob per token: {log_probs.mean().item():.4f}')

In [ ]:
# Compute reward for this response
full_text = prompt + ' ' + response
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

enc = tokenizer(
    full_text,
    return_tensors='pt',
    max_length=256,
    truncation=True,
    padding='max_length',
).to(device)

reward_model.eval()
with torch.no_grad():
    reward = reward_model(enc['input_ids'], enc['attention_mask'])

print(f'Reward for generated response: {reward.item():.4f}')

## Step 7: Evaluation Metrics

In [ ]:
from eval.metrics import RewardModelAccuracy, KLDivergenceTracker, RewardStats

# Demo the metrics
accuracy_metric = RewardModelAccuracy()
kl_tracker = KLDivergenceTracker(window_size=100)
reward_stats = RewardStats()

# Simulate some evaluation
import random
random.seed(42)

for _ in range(50):
    # Simulate reward pairs
    r_chosen = torch.tensor([random.gauss(0.3, 0.2) for _ in range(4)])
    r_rejected = torch.tensor([random.gauss(-0.1, 0.2) for _ in range(4)])
    
    accuracy_metric.update(r_chosen, r_rejected)
    reward_stats.update(r_chosen)
    
    # Simulate KL
    kl = torch.tensor(random.gauss(0.08, 0.02))
    kl_tracker.update(kl)

print('=== Evaluation Summary ===')
print(f'Reward Model Accuracy: {accuracy_metric.compute():.4f}')
print(f'KL Divergence (mean):  {kl_tracker.mean():.4f}')
print(f'KL Divergence (std):   {kl_tracker.std():.4f}')

reward_summary = reward_stats.compute()
print('\nReward Statistics:')
for k, v in reward_summary.items():
    print(f'  {k}: {v:.4f}')

## Step 8: Ablation Study Overview

The `eval/ablation.py` script sweeps `kl_coef` over `[0.01, 0.05, 0.1, 0.2, 0.5]`
and measures the reward-KL tradeoff.

**Key insight:** Increasing `kl_coef` reduces the KL divergence (policy stays close
to reference) but also reduces the achievable reward. The optimal β typically
falls in `[0.05, 0.1]` for GPT-scale models.

To run the full ablation:
```bash
python -m eval.ablation --config configs/config.yaml --n-steps 50
```

In [ ]:
# Visualize expected ablation results (simulated for illustration)
import numpy as np

kl_coefs = [0.01, 0.05, 0.1, 0.2, 0.5]
# Simulated: reward decreases with kl_coef, kl decreases with kl_coef
sim_rewards = [0.41, 0.39, 0.37, 0.33, 0.28]
sim_kl = [0.89, 0.42, 0.20, 0.09, 0.03]

fig, ax1 = plt.subplots(figsize=(8, 5))

color_r = '#2196F3'
color_k = '#F44336'

ax1.plot(kl_coefs, sim_rewards, 'o-', color=color_r, linewidth=2, markersize=8, label='Reward')
ax1.set_xlabel('KL Coefficient (β)', fontsize=12)
ax1.set_ylabel('Final Reward', color=color_r, fontsize=12)
ax1.tick_params(axis='y', labelcolor=color_r)

ax2 = ax1.twinx()
ax2.plot(kl_coefs, sim_kl, 's--', color=color_k, linewidth=2, markersize=8, label='KL Div')
ax2.set_ylabel('KL Divergence', color=color_k, fontsize=12)
ax2.tick_params(axis='y', labelcolor=color_k)

plt.title('Reward vs KL Coefficient Tradeoff (Simulated)', fontsize=13)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

print('Ablation Summary Table:')
print(f'{"β":>8} | {"Reward":>8} | {"KL":>8}')
print('-' * 32)
for b, r, k in zip(kl_coefs, sim_rewards, sim_kl):
    print(f'{b:>8.3f} | {r:>8.4f} | {k:>8.4f}')

## Summary

We walked through the complete RLHF pipeline:

| Step | Module | Key Concept |
|------|--------|-------------|
| 1. Collect | `PreferenceDataCollector` | Heuristic scoring → (prompt, chosen, rejected) JSONL |
| 2. Dataset | `PreferenceDataset` | HuggingFace tokenization → tensors |
| 3. Reward Model | `RewardModel` | GPT-2 + linear head, Bradley-Terry loss |
| 4. Policy | `PolicyModel` | CausalLM wrapper, log probs, KL computation |
| 5. PPO | `PPOTrainer` | Clipped surrogate + KL penalty |
| 6. Eval | `metrics.py` | Accuracy, KL tracker, reward stats |
| 7. Ablation | `ablation.py` | Sweep β, measure reward-KL tradeoff |

For production use, replace the heuristic scorer with real human annotations or a stronger oracle model.